## 1. Imports & Load Cleaned Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

PROCESSED_DIR = Path('../data/processed')
OUT_DIR = Path('../data/features')
OUT_DIR.mkdir(parents=True, exist_ok=True)

results_df    = pd.read_parquet(PROCESSED_DIR / 'results.parquet')
races_df      = pd.read_parquet(PROCESSED_DIR / 'races.parquet')
drivers_df    = pd.read_parquet(PROCESSED_DIR / 'drivers.parquet')
constructors_df = pd.read_parquet(PROCESSED_DIR / 'constructors.parquet')
pit_stops_df  = pd.read_parquet(PROCESSED_DIR / 'pit_stops.parquet')
lap_times_df  = pd.read_parquet(PROCESSED_DIR / 'lap_times.parquet')
qualifying_df = pd.read_parquet(PROCESSED_DIR / 'qualifying.parquet')

# Bring Season/race_date onto results so everything below can be ordered/grouped by it
results_df = results_df.merge(
    races_df[['raceId', 'Season', 'Round', 'race_date', 'circuitId']],
    on='raceId', how='left'
)

results_df[['raceId', 'driverId', 'constructorId', 'Season', 'points', 'grid', 'FinishPosition']].head()


,raceId,driverId,constructorId,Season,points,grid,FinishPosition
0,18,1,1,2008,10.0,1.0,1.0
1,18,2,2,2008,8.0,5.0,2.0
2,18,3,3,2008,6.0,7.0,3.0
3,18,4,4,2008,5.0,11.0,4.0
4,18,5,1,2008,4.0,3.0,5.0


## 2. Teammate Pairing & Performance Deltas

For every driver-season, identify their constructor teammate(s) — handling mid-season
driver changes, where a driver may have more than one teammate across a season — and
compute `TeammatePointsDelta`, `TeammateGridDelta`, and `TeammateQualifyingDelta`. This
is the cleanest driver evaluation signal in the dataset because both drivers share the
same car, so any delta is attributable to the driver rather than the machinery (see
KPI #14/#15).

**Two separate qualifying-related deltas, deliberately kept distinct:**
- `TeammateGridDelta` — based on `FactRaceResults[grid]` (actual starting position,
  post-penalty). Useful for race-day starting-position context.
- `TeammateQualifyingDelta` — based on `FactQualifying[QualifyingPosition]` (raw
  Saturday session result, pre-penalty). This is the one that isolates true one-lap
  pace, per KPI #15 — use this one, not the grid-based one, for pure pace comparisons.

In [3]:
# Season-level points and average grid position per driver, per constructor stint
driver_season_summary = (
    results_df
    .groupby(['Season', 'constructorId', 'driverId'])
    .agg(
        SeasonPoints=('points', 'sum'),
        AvgGridPosition=('grid', 'mean'),
        RacesForConstructor=('raceId', 'nunique'),
    )
    .reset_index()
)

qualifying_with_season = qualifying_df.merge(
    races_df[['raceId', 'Season']], on='raceId', how='left'
)
avg_quali_position = (
    qualifying_with_season
    .groupby(['Season', 'driverId'])['QualifyingPosition']
    .mean()
    .rename('AvgQualifyingPosition')
    .reset_index()
)

driver_season_summary = driver_season_summary.merge(
    avg_quali_position, on=['Season', 'driverId'], how='left'
)

driver_season_summary.head()


,Season,constructorId,driverId,SeasonPoints,AvgGridPosition,RacesForConstructor,AvgQualifyingPosition
0,1950,6,633,0.0,4.666667,3,NaN
1,1950,6,647,8.0,5.250000,4,NaN
2,1950,6,687,4.0,19.000000,3,NaN
3,1950,6,791,0.0,25.000000,1,NaN
4,1950,6,793,3.0,11.000000,2,NaN


In [4]:
def compute_teammate_deltas(group):
    """Within a single Season+constructor group, compare each driver against the
    average of all *other* drivers who drove for that constructor that season —
    this correctly handles the 3+ driver case (mid-season replacements) rather than
    assuming exactly two teammates. Computes both a grid-based delta (post-penalty
    starting position) and a qualifying-based delta (raw one-lap pace) — see the
    distinction explained in the markdown cell above."""
    group = group.copy()
    n = len(group)
    if n < 2:
        group['TeammatePoints'] = np.nan
        group['TeammatePointsDelta'] = np.nan
        group['TeammateAvgGridPosition'] = np.nan
        group['TeammateGridDelta'] = np.nan
        group['TeammateAvgQualifyingPosition'] = np.nan
        group['TeammateQualifyingDelta'] = np.nan
        return group

    total_points = group['SeasonPoints'].sum()
    total_grid = group['AvgGridPosition'].sum()
    total_quali = group['AvgQualifyingPosition'].sum()

    group['TeammatePoints'] = (total_points - group['SeasonPoints']) / (n - 1)
    group['TeammatePointsDelta'] = group['SeasonPoints'] - group['TeammatePoints']

    group['TeammateAvgGridPosition'] = (total_grid - group['AvgGridPosition']) / (n - 1)
    # Negative delta = better (lower) average starting position than the teammate
    group['TeammateGridDelta'] = group['AvgGridPosition'] - group['TeammateAvgGridPosition']

    group['TeammateAvgQualifyingPosition'] = (total_quali - group['AvgQualifyingPosition']) / (n - 1)
    # Negative delta = better (lower) raw qualifying position than the teammate —
    # this is the one that reflects pure one-lap pace, unaffected by grid penalties
    group['TeammateQualifyingDelta'] = group['AvgQualifyingPosition'] - group['TeammateAvgQualifyingPosition']

    return group

teammate_deltas = (
    driver_season_summary
    .groupby(['Season', 'constructorId'], group_keys=False)
    .apply(compute_teammate_deltas)
)

teammate_deltas.sort_values('TeammatePointsDelta', ascending=False).head(10)


C:\Users\harut\AppData\Local\Temp\ipykernel_29792\1608056852.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_teammate_deltas)


,Season,constructorId,driverId,SeasonPoints,AvgGridPosition,RacesForConstructor,AvgQualifyingPosition,TeammatePoints,TeammatePointsDelta,TeammateAvgGridPosition,TeammateGridDelta,TeammateAvgQualifyingPosition,TeammateQualifyingDelta
3546,2025,9,830,389.0,3.608696,24,3.500000,10.5,378.5,16.095238,-12.486542,12.562500,-9.062500
3500,2023,9,830,530.0,3.181818,22,3.000000,260.0,270.0,7.454545,-4.272727,9.090909,-6.090909
3524,2024,9,830,399.0,3.541667,24,2.916667,138.0,261.0,8.541667,-5.000000,9.333333,-6.416667
3439,2020,131,1,347.0,1.875000,16,1.687500,113.0,234.0,2.147059,-0.272059,8.529412,-6.841912
3415,2019,9,830,278.0,4.714286,21,4.571429,69.5,208.5,6.847222,-2.132937,10.203571,-5.632143
3457,2021,9,830,388.5,2.863636,22,2.818182,190.0,198.5,4.409091,-1.545455,6.409091,-3.590909
3272,2013,9,20,397.0,2.052632,19,2.052632,199.0,198.0,5.421053,-3.368421,4.421053,-2.368421
3521,2024,6,844,327.0,5.416667,24,5.333333,134.0,193.0,7.913043,-2.496377,9.260870,-3.927536
3463,2021,131,1,385.5,3.090909,22,2.136364,219.0,166.5,5.590909,-2.500000,3.772727,-1.636364
3400,2018,131,1,408.0,2.714286,21,2.476190,247.0,161.0,4.142857,-1.428571,3.523810,-1.047619


In [5]:
# teammate's points in the rookie's debut season (Business Question #28)

first_season = results_df.groupby('driverId')['Season'].min().rename('DebutSeason')
teammate_deltas = teammate_deltas.merge(first_season, on='driverId', how='left') \
    if 'driverId' in teammate_deltas.columns else teammate_deltas

if 'driverId' not in teammate_deltas.columns:
    teammate_deltas = teammate_deltas.reset_index()

teammate_deltas['IsDebutSeason'] = teammate_deltas['Season'] == teammate_deltas['DebutSeason']
teammate_deltas['RookieSeasonIndex'] = np.where(
    teammate_deltas['IsDebutSeason'] & (teammate_deltas['TeammatePoints'] > 0),
    teammate_deltas['SeasonPoints'] / teammate_deltas['TeammatePoints'],
    np.nan
)

teammate_deltas[teammate_deltas['IsDebutSeason']].sort_values(
    'RookieSeasonIndex', ascending=False
).head(10)


,Season,constructorId,driverId,SeasonPoints,AvgGridPosition,RacesForConstructor,AvgQualifyingPosition,TeammatePoints,TeammatePointsDelta,TeammateAvgGridPosition,TeammateGridDelta,TeammateAvgQualifyingPosition,TeammateQualifyingDelta,DebutSeason,IsDebutSeason,RookieSeasonIndex
35,1950,113,593,9.0,5.000000,1,NaN,0.307692,8.692308,15.307692,-10.307692,NaN,NaN,1950,True,29.250000
189,1952,87,578,10.0,8.600000,5,NaN,0.400000,9.600000,14.400000,-5.800000,NaN,NaN,1952,True,25.000000
1827,1974,37,233,5.0,13.000000,14,NaN,0.200000,4.800000,12.900000,0.100000,NaN,NaN,1974,True,25.000000
65,1950,154,627,13.0,9.333333,6,NaN,0.636364,12.363636,11.962121,-2.628788,NaN,NaN,1950,True,20.428571
16,1950,105,669,5.0,10.750000,4,NaN,0.352941,4.647059,16.258824,-5.508824,NaN,NaN,1950,True,14.166667
2370,1984,53,102,13.0,10.000000,15,NaN,1.000000,12.000000,11.188889,-1.188889,NaN,NaN,1984,True,13.000000
2711,1993,17,22,2.0,14.187500,16,NaN,0.200000,1.800000,20.640000,-6.452500,NaN,NaN,1993,True,10.000000
1923,1975,75,178,2.0,21.500000,4,NaN,0.200000,1.800000,16.222222,5.277778,NaN,NaN,1975,True,10.000000
14,1950,105,589,4.0,13.400000,5,NaN,0.411765,3.588235,16.102941,-2.702941,NaN,NaN,1950,True,9.714286
588,1955,132,608,6.0,5.666667,3,NaN,0.666667,5.333333,10.000000,-4.333333,NaN,NaN,1955,True,9.000000


## 3. Stint Length & Lap Grouping

Derive stint boundaries per driver-race from `pit_stops` + `lap_times` — the lap ranges
between stops, which feed both the tire-strategy visuals on Dashboard Page 6 and the
degradation-slope model in `advanced_analytics.ipynb`.


In [6]:
def build_stints_for_race_driver(race_id, driver_id, pit_stops_df, lap_times_df):
    """Returns a DataFrame of stints (StintNumber, StartLap, EndLap, LapCount) for a
    single driver in a single race, derived from that driver's pit stop laps."""
    stops = pit_stops_df[
        (pit_stops_df['raceId'] == race_id) & (pit_stops_df['driverId'] == driver_id)
    ].sort_values('lap')

    laps = lap_times_df[
        (lap_times_df['raceId'] == race_id) & (lap_times_df['driverId'] == driver_id)
    ]
    if laps.empty:
        return pd.DataFrame()

    max_lap = laps['lap'].max()
    stop_laps = stops['lap'].tolist()

    boundaries = [0] + stop_laps + [max_lap]
    stints = []
    for i in range(len(boundaries) - 1):
        start_lap = boundaries[i] + 1 if i > 0 else 1
        end_lap = boundaries[i + 1]
        if end_lap < start_lap:
            continue
        stints.append({
            'raceId': race_id,
            'driverId': driver_id,
            'StintNumber': i + 1,
            'StartLap': start_lap,
            'EndLap': end_lap,
            'LapCount': end_lap - start_lap + 1,
        })
    return pd.DataFrame(stints)

all_pairs = lap_times_df[['raceId', 'driverId']].drop_duplicates()

stint_frames = []
for race_id, driver_id in all_pairs.itertuples(index=False):
    stint_frames.append(
        build_stints_for_race_driver(race_id, driver_id, pit_stops_df, lap_times_df)
    )

stints_df = pd.concat(stint_frames, ignore_index=True) if stint_frames else pd.DataFrame()
stints_df.head(10)


,raceId,driverId,StintNumber,StartLap,EndLap,LapCount
0,479,137,1,1,31,31
1,479,119,1,1,27,27
2,479,105,1,1,1,1
3,479,205,1,1,32,32
4,479,177,1,1,53,53
5,479,187,1,1,44,44
6,479,182,1,1,52,52
7,479,181,1,1,15,15
8,479,173,1,1,53,53
9,479,95,1,1,17,17


In [7]:
# Attach average lap time per stint

def avg_stint_lap_time(row, lap_times_df):
    laps = lap_times_df[
        (lap_times_df['raceId'] == row['raceId']) &
        (lap_times_df['driverId'] == row['driverId']) &
        (lap_times_df['lap'] >= row['StartLap']) &
        (lap_times_df['lap'] <= row['EndLap'])
    ]
    return laps['LapTimeMs'].mean()

if not stints_df.empty:
    stints_df['AvgStintLapTimeMs'] = stints_df.apply(
        lambda r: avg_stint_lap_time(r, lap_times_df), axis=1
    )

stints_df.head(10)


,raceId,driverId,StintNumber,StartLap,EndLap,LapCount,AvgStintLapTimeMs
0,479,137,1,1,31,31,97231.258065
1,479,119,1,1,27,27,95938.629630
2,479,105,1,1,1,1,107893.000000
3,479,205,1,1,32,32,99203.687500
4,479,177,1,1,53,53,96269.094340
5,479,187,1,1,44,44,100850.931818
6,479,182,1,1,52,52,99407.519231
7,479,181,1,1,15,15,102304.133333
8,479,173,1,1,53,53,96268.150943
9,479,95,1,1,17,17,119693.117647


## 4. Rolling Form Features

5-race rolling average points and positions gained, per driver, ordered by race date.
Validated here before being re-expressed as DAX time-intelligence measures (#39–41) for
interactive, slicer-responsive use in the report — this Python version exists to sanity
check that DAX output against an independent calculation, and to feed the forecasting
model in `advanced_analytics.ipynb`.


In [8]:
results_sorted = results_df.sort_values(['driverId', 'race_date']).copy()

results_sorted['Rolling5RaceAvgPoints'] = (
    results_sorted
    .groupby('driverId')['points']
    .transform(lambda s: s.rolling(window=5, min_periods=1).mean())
)

results_sorted['Rolling5RaceAvgPositionsGained'] = (
    results_sorted
    .groupby('driverId')['PositionsGained']
    .transform(lambda s: s.rolling(window=5, min_periods=1).mean())
)

results_sorted[[
    'driverId', 'raceId', 'race_date', 'points',
    'Rolling5RaceAvgPoints', 'PositionsGained', 'Rolling5RaceAvgPositionsGained'
]].head(10)


,driverId,raceId,race_date,points,Rolling5RaceAvgPoints,PositionsGained,Rolling5RaceAvgPositionsGained
370,1,36,2007-03-18,6.0,6.000000,1.0,1.00
391,1,37,2007-04-08,8.0,7.000000,2.0,1.50
413,1,38,2007-04-15,8.0,7.333333,0.0,1.00
435,1,39,2007-05-13,8.0,7.500000,2.0,1.25
457,1,40,2007-05-27,8.0,7.600000,0.0,1.00
478,1,41,2007-06-10,10.0,8.400000,0.0,0.80
500,1,42,2007-06-17,10.0,8.800000,0.0,0.40
524,1,43,2007-07-01,6.0,8.400000,-1.0,0.20
546,1,44,2007-07-08,6.0,8.000000,-2.0,-0.60
574,1,45,2007-07-22,0.0,6.400000,1.0,-0.40


## 5. Export Feature Tables

Exported separately from the core cleaned tables so the Power BI model can merge them
in as extra columns on `FactRaceResults` / a new stints table, without re-running any
of the logic above inside Power Query.


In [9]:
teammate_deltas.to_parquet(OUT_DIR / 'teammate_deltas.parquet', index=False)
stints_df.to_parquet(OUT_DIR / 'stints.parquet', index=False)
results_sorted[[
    'raceId', 'driverId', 'Rolling5RaceAvgPoints', 'Rolling5RaceAvgPositionsGained'
]].to_parquet(OUT_DIR / 'rolling_form.parquet', index=False)

print("Feature tables written to", OUT_DIR)
for f in OUT_DIR.glob('*.parquet'):
    print(" -", f.name)

print("\nDone — ready for advanced_analytics.ipynb")


Feature tables written to ..\data\features
 - rolling_form.parquet
 - stints.parquet
 - teammate_deltas.parquet

Done — ready for advanced_analytics.ipynb
